> **Kernel:** run this notebook with the **`venv-torch2.3`** Jupyter kernel (torch 2.3.1) — pick it from the kernel picker before running any cell.

# Multimodal Pruning + LoRA Recovery (T4 Demo)
This notebook demonstrates:
- **Model:** Qwen/Qwen2-VL-2B-Instruct, a multimodal model for text and vision tasks.
- **Pruning:** Logical masking to prune 30% of attention heads or FFN channels.
- **Fine-tuning:** LoRA (Low-Rank Adaptation) applied to attention and MLP layers to recover accuracy.
- **Evaluation:** Toy perplexity and generation tasks on colored square images.

# 1) Setup (Installs & GPU Check)
This cell:
- Sets up the environment for the notebook.
- Prints the versions of PyTorch, CUDA, and Python.
- Checks if a GPU is available and prints its name.
- Sets a random seed for reproducibility.

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # or "true" if you prefer

import os, gc, math, random, platform, warnings
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import List

print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| Py:", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# Repro
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

Torch: 2.3.1+cu121 | CUDA: 12.1 | Py: 3.10.14
GPU: Tesla T4


# 2) Imports & Configuration
This cell:
- Imports libraries for image processing, model configuration, and training.
- Defines key configuration parameters:
  - `MODEL_NAME`: The name of the model to load.
  - `PRUNE_MODE`: The pruning strategy (attention heads or FFN channels).
  - `PRUNE_RATIO`: The percentage of components to prune.
  - `USE_FP32`: Whether to use FP32 precision (default is FP16 for T4 GPUs).
- Sets learning rate, batch size, and other training parameters.
- Creates an output directory for saving results.

In [ ]:
from PIL import Image
import torchvision
import torchvision.transforms as T

from transformers import (
    AutoProcessor, AutoConfig, AutoModelForVision2Seq, get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model

# --- Demo knobs ---
MODEL_NAME   = "Qwen/Qwen2-VL-2B-Instruct"
PRUNE_MODE   = "ffn_channels"        # "attn_heads" (GQA-safe) or "ffn_channels"
PRUNE_RATIO  = 0.15                  # try 0.15-0.30
USE_FP32     = False                 # bfloat16 recommended: same memory as FP16 (2 B/param) but float32
                            # exponent range (max ~3.4e38 vs 65504 in FP16).
                            # Prevents overflow in pruned-model logits.

# LoRA / Train knobs (tiny)
LR                = 1e-4
EPOCHS            = 1
BATCH_SIZE        = 1       # was 2: with 224px images, BATCH_SIZE=2 puts 2x256=512
                            # ViT patches through attention together. Keeping it at 1
                            # further halves this cost. Effective batch is unchanged
                            # because GRAD_ACCUM_STEPS=4 -> effective batch = 4.
GRAD_ACCUM_STEPS  = 4
MAX_LENGTH        = 256     # was 512: VQA answers are 1-3 words; shorter sequences
                            # reduce LLM decoder activation memory noticeably.
WARMUP_STEPS      = 20
N_TRAIN, N_VAL    = 800, 200

OUTPUT_DIR        = "./demo_pruned_lora_qwen2vl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.float32 if USE_FP32 else torch.bfloat16
print(f"Device: {device} | Dtype: {dtype} | Prune: {PRUNE_MODE} @ {int(PRUNE_RATIO*100)}%")


# 3) Load Model & Processor
This cell:
- Loads the processor and configuration for the specified model.
- Defines a helper function to load the model with the desired precision (FP16 or FP32).
- Enables gradient checkpointing to save memory during training.
- Extracts key model dimensions such as hidden size, number of attention heads, and intermediate size.
- Prints the total number of parameters in the model.

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
config    = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)

def load_model(dtype):
    try:
        m = AutoModelForVision2Seq.from_pretrained(
            MODEL_NAME, torch_dtype=dtype, low_cpu_mem_usage=True, trust_remote_code=True
        ).to(device)
        return m, dtype
    except torch.cuda.OutOfMemoryError:
        print("OOM at requested dtype; falling back to FP16.")
        torch.cuda.empty_cache(); gc.collect()
        m = AutoModelForVision2Seq.from_pretrained(
            MODEL_NAME, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
        ).to(device)
        return m, torch.float16

model, dtype = load_model(dtype)
model.gradient_checkpointing_enable()  # memory saver

hidden_size        = getattr(config, "hidden_size", getattr(config, "hidden_dim", None))
num_heads          = getattr(config, "num_attention_heads", getattr(config, "num_heads", None))
intermediate_size  = getattr(config, "intermediate_size", getattr(config, "ffn_hidden_size", None))
assert hidden_size and num_heads and intermediate_size, "Missing key config dims."
head_dim = hidden_size // num_heads

total_params_m = sum(p.numel() for p in model.parameters())/1e6
print(f"Total params: {total_params_m:.1f}M")

`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Total params: 2209.0M


# 4) Module Finders (Attention & MLP)
This cell:
- Defines utility functions to locate specific modules in the model:
  - `find_attn_modules`: Finds attention modules with Q, K, V, and O projections.
  - `find_mlp_modules`: Finds MLP modules with gate, up, and down projections.
- These functions are used to identify modules for pruning and fine-tuning.

In [4]:
def find_attn_modules(module):
    for name, m in module.named_modules():
        if all(hasattr(m, x) for x in ["q_proj","k_proj","v_proj","o_proj"]):
            yield name, m

def find_mlp_modules(module):
    for name, m in module.named_modules():
        if all(hasattr(m, x) for x in ["gate_proj","up_proj","down_proj"]):
            yield name, m


# 5) Pruning Utilities (Logical Masking)
This cell:
- Implements pruning utilities for attention heads and FFN channels:
  - `prune_attention_heads_logical_gqa`: Prunes attention heads while maintaining GQA (Grouped Query Attention) constraints.
  - `prune_ffn_channels_logical_mask`: Prunes FFN channels by masking weights based on their norms.
- These functions apply logical masking to avoid altering the model's architecture.

In [5]:
@torch.no_grad()
def prune_attention_heads_logical_gqa(attn_mod, ratio: float):
    n_q  = getattr(attn_mod, "num_heads", None) \
        or getattr(attn_mod, "n_heads", None) \
        or getattr(getattr(attn_mod, "config", None), "num_attention_heads", None)
    n_kv = getattr(attn_mod, "num_key_value_heads", None) \
        or getattr(attn_mod, "n_kv_heads", None) \
        or getattr(getattr(attn_mod, "config", None), "num_key_value_heads", None) \
        or n_q
    hd = getattr(attn_mod, "head_dim", None)
    if hd is None:
        q_rows = attn_mod.q_proj.weight.shape[0]
        assert n_q and q_rows % n_q == 0, "Cannot infer head_dim."
        hd = q_rows // n_q
    assert n_q and n_kv and n_q >= 1 and n_kv >= 1
    assert n_q % n_kv == 0, "Expected n_q divisible by n_kv for GQA."

    n_keep_q = max(1, int(n_q * (1.0 - ratio)))
    prune_q  = list(range(n_keep_q, n_q))
    if not prune_q:
        return

    def rows_for_heads(head_ids, per_head):
        rows = []
        for h in head_ids:
            s = h * per_head
            rows.extend(range(s, s + per_head))
        return rows

    rows_q  = rows_for_heads(prune_q, hd)
    group   = max(1, n_q // n_kv)
    prune_kv = sorted(set(h // group for h in prune_q))
    rows_kv = rows_for_heads(prune_kv, hd)

    # q_proj rows -> mask multiply
    Wq = attn_mod.q_proj.weight
    mask_q = torch.ones(Wq.shape[0], device=Wq.device, dtype=Wq.dtype)
    if rows_q:
        idx_q = torch.tensor([r for r in rows_q if 0 <= r < Wq.shape[0]], device=Wq.device, dtype=torch.long)
        if idx_q.numel() > 0:
            mask_q.index_fill_(0, idx_q, 0)
            Wq.mul_(mask_q[:, None])
            bq = getattr(attn_mod.q_proj, "bias", None)
            if bq is not None: bq.mul_(mask_q.to(bq.dtype))

    # k_proj / v_proj rows
    for pname, rows in (("k_proj", rows_kv), ("v_proj", rows_kv)):
        proj = getattr(attn_mod, pname)
        W    = proj.weight
        mask = torch.ones(W.shape[0], device=W.device, dtype=W.dtype)
        if rows:
            idx  = torch.tensor([r for r in rows if 0 <= r < W.shape[0]], device=W.device, dtype=torch.long)
            if idx.numel() > 0:
                mask.index_fill_(0, idx, 0)
                W.mul_(mask[:, None])
                b = getattr(proj, "bias", None)
                if b is not None: b.mul_(mask.to(b.dtype))

    # o_proj columns corresponding to pruned Q rows
    Wo = attn_mod.o_proj.weight
    col_mask = torch.ones(Wo.shape[1], device=Wo.device, dtype=Wo.dtype)
    if rows_q:
        idx_cols = torch.tensor([c for c in rows_q if 0 <= c < Wo.shape[1]], device=Wo.device, dtype=torch.long)
        if idx_cols.numel() > 0:
            col_mask.index_fill_(0, idx_cols, 0)
            Wo.mul_(col_mask[None, :])

In [6]:
@torch.no_grad()
def prune_ffn_channels_logical_mask(mlp_mod: nn.Module, ratio: float, intermediate_size: int):
    n_prune = max(1, int(intermediate_size * ratio))
    if n_prune <= 0: return
    down_W = mlp_mod.down_proj.weight  # [hidden, inter]
    col_norms = torch.norm(down_W, p=1, dim=0)
    prune_idx = torch.topk(col_norms, k=n_prune, largest=False).indices

    inter_dim = down_W.shape[1]
    keep_mask = torch.ones(inter_dim, device=down_W.device, dtype=down_W.dtype)
    if prune_idx.numel() > 0:
        keep_mask.index_fill_(0, prune_idx, 0)

    # down_proj: zero columns
    mlp_mod.down_proj.weight.mul_(keep_mask[None, :])
    # up_proj / gate_proj: zero rows
    for name in ["up_proj","gate_proj"]:
        getattr(mlp_mod, name).weight.mul_(keep_mask[:, None])
        b = getattr(getattr(mlp_mod, name), "bias", None)
        if b is not None:
            b.mul_(keep_mask.to(b.dtype))

# 6) Build a Tiny Multimodal Toy Dataset
This cell:
- Prepares a toy dataset using CIFAR-10 images and their labels.
- Resizes images to 448x448 pixels to match the model's input size.
- Creates training and validation datasets with a fixed number of examples.
- Each example includes:
  - An image.
  - A question ("What is in this image?").
  - The answer (e.g., "cat") with and without an EOS token.

In [7]:
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
PAD_ID   = processor.tokenizer.pad_token_id
IGNOREID = -100
EOS      = processor.tokenizer.eos_token

root = "./data_cifar10"
train_raw = torchvision.datasets.CIFAR10(root=root, train=True, download=True)
test_raw  = torchvision.datasets.CIFAR10(root=root, train=False, download=True)
label_names = train_raw.classes

# 224 instead of 448: patches per image drop from 1024 to 256 (4x fewer).
# ViT attention scores scale as O(patches^2), so the vision encoder uses
# 16x less memory per layer. Also cuts CPU RAM for the encoded dataset ~4x.
resize_to = 224
resize_tf = T.Resize((resize_to, resize_to))

def make_examples(ds, n_take):
    items = []
    for i in range(n_take):
        img_pil, y = ds[i]
        img = resize_tf(img_pil)
        ans = label_names[y]
        items.append({
            "image": img,
            "question": "What is in this image?",
            "answer": ans,                  # clean answer
            "answer_with_eos": ans + EOS,   # teach the model to STOP
        })
    return items

N_TRAIN, N_VAL = 800, 200
train_items = make_examples(train_raw, N_TRAIN)
val_items   = make_examples(test_raw,  N_VAL)

# Free the raw CIFAR-10 datasets from CPU RAM — they are no longer needed
# (only the resized PIL images in train_items/val_items are used going forward).
del train_raw, test_raw
import gc; gc.collect()


Files already downloaded and verified


Files already downloaded and verified


21

# Encode Dataset for VQA
This cell:
- Encodes the dataset for visual question answering (VQA) tasks.
- Defines a function to tokenize and preprocess each example:
  - Constructs training and generation templates.
  - Masks labels to supervise only the answer tokens.
- Creates PyTorch datasets for training and validation.

In [8]:
import hashlib, pickle, pathlib, time

def encode_example_vqa(ex):
    # Train template: user + assistant(answer_with_eos)
    messages_train = [
        {"role":"user","content":[
            {"type":"image","image": ex["image"]},
            {"type":"text","text": ex["question"]}
        ]},
        {"role":"assistant","content":[{"type":"text","text": ex["answer_with_eos"]}]}
    ]
    # Generation template: user only (+ generation prompt)
    messages_gen = [
        {"role":"user","content":[
            {"type":"image","image": ex["image"]},
            {"type":"text","text": ex["question"]}
        ]}
    ]
    train_text = processor.apply_chat_template(messages_train, tokenize=False, add_generation_prompt=False)
    gen_text   = processor.apply_chat_template(messages_gen,   tokenize=False, add_generation_prompt=True)

    out = processor(
        text=[train_text],
        images=[ex["image"]],
        return_tensors="pt",
        max_length=MAX_LENGTH,
        padding="longest",
        truncation=True
    )

    # Supervise only the answer_with_eos tokens (exact span match in tokenized sequence)
    def mask_answer_only(input_ids_2d, answer_text):
        full_ids = input_ids_2d[0].tolist()
        ans_ids  = processor.tokenizer(answer_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0].tolist()

        def find_subseq(a, b):
            L, M = len(a), len(b)
            if M == 0 or M > L: return -1
            for i in range(L - M + 1):
                if a[i:i+M] == b: return i
            return -1

        labels = input_ids_2d.clone(); labels[:] = -100
        start = find_subseq(full_ids, ans_ids)
        if start >= 0:
            end = start + len(ans_ids)
            labels[:, start:end] = input_ids_2d[:, start:end]
        else:
            # fallback: supervise last few tokens
            keep = min(8, input_ids_2d.shape[1])
            labels[:, -keep:] = input_ids_2d[:, -keep:]
        return labels

    input_ids = out["input_ids"]
    labels    = mask_answer_only(input_ids, ex["answer_with_eos"])

    return {
        "pixel_values": out["pixel_values"].squeeze(0),
        "image_grid_thw": out.get("image_grid_thw", None).squeeze(0) if out.get("image_grid_thw", None) is not None else None,
        "input_ids": input_ids.squeeze(0),
        "attention_mask": out["attention_mask"].squeeze(0),
        "labels": labels.squeeze(0),
        # for eval
        "answer_text": ex["answer"],
        "raw_image":   ex["image"],
        "gen_prompt":  gen_text,
    }

# ── Dataset encoding cache ────────────────────────────────────────────────────
# Encoding 1000 images through the processor is the slowest CPU step (~minutes).
# We cache the result to disk keyed on all parameters that affect the output.
# On subsequent kernel restarts the cache is loaded in seconds instead.
_cache_key = hashlib.md5(
    f"{MODEL_NAME}|{resize_to}|{MAX_LENGTH}|{N_TRAIN}|{N_VAL}".encode()
).hexdigest()[:12]
_cache_path = pathlib.Path(f"./data_cifar10/encoded_cache_{_cache_key}.pkl")

if _cache_path.exists():
    print(f"Loading encoded dataset from cache ({_cache_path.name}) ...")
    t0 = time.time()
    with open(_cache_path, "rb") as _f:
        train_encoded, val_encoded = pickle.load(_f)
    print(f"Loaded {len(train_encoded)} train + {len(val_encoded)} val examples "
          f"in {time.time()-t0:.1f}s  (cache hit)")
else:
    print(f"Encoding {N_TRAIN} train + {N_VAL} val examples (first run — will be cached) ...")
    t0 = time.time()
    train_encoded = [encode_example_vqa(ex) for ex in train_items]
    val_encoded   = [encode_example_vqa(ex) for ex in val_items]
    with open(_cache_path, "wb") as _f:
        pickle.dump((train_encoded, val_encoded), _f)
    print(f"Encoded and cached to {_cache_path.name} in {time.time()-t0:.1f}s")
    print(f"  Cache size: {_cache_path.stat().st_size / 1024**2:.0f} MB  "
          f"(future runs will skip encoding)")

class CIFARVQADataset(Dataset):
    def __init__(self, encoded): self.encoded = encoded
    def __len__(self): return len(self.encoded)
    def __getitem__(self, i): return self.encoded[i]

train_ds = CIFARVQADataset(train_encoded)
val_ds   = CIFARVQADataset(val_encoded)


Encoding 800 train + 200 val examples (first run — will be cached) ...


Encoded and cached to encoded_cache_c166508947d4.pkl in 6.5s
  Cache size: 1296 MB  (future runs will skip encoding)


# DataLoader with Collation
This cell:
- Defines a collation function to pad sequences and batch examples.
- Creates PyTorch DataLoaders for training and validation datasets.
- Ensures that all inputs (e.g., pixel values, input IDs) are properly batched.

In [9]:
def pad_1d(seqs, pad_val):
    maxlen = max(x.size(0) for x in seqs)
    out = torch.full((len(seqs), maxlen), pad_val, dtype=seqs[0].dtype)
    for i, s in enumerate(seqs):
        out[i, :s.size(0)] = s
    return out

def collate_fn(batch):
    out = {}
    out["pixel_values"]   = torch.stack([b["pixel_values"] for b in batch], dim=0)
    grids = [b["image_grid_thw"] for b in batch]
    out["image_grid_thw"] = torch.stack(grids, dim=0) if all(g is not None for g in grids) else None

    out["input_ids"]      = pad_1d([b["input_ids"] for b in batch], PAD_ID)
    out["attention_mask"] = pad_1d([b["attention_mask"] for b in batch], 0)
    out["labels"]         = pad_1d([b["labels"] for b in batch], IGNOREID)

    out["answer_text"] = [b["answer_text"] for b in batch]
    out["raw_images"]  = [b["raw_image"]   for b in batch]
    out["gen_prompts"] = [b["gen_prompt"]  for b in batch]
    return out

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=2,          shuffle=False, collate_fn=collate_fn)

# 7) Baseline Toy Perplexity
This cell:
- Evaluates the model's perplexity on the training dataset before pruning.
- Defines a function to compute perplexity by averaging the loss over tokens.
- Evaluates the model's generation accuracy on the validation dataset.

In [10]:
@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for batch in loader:
        moved = {}
        for k, v in batch.items():
            if k == "pixel_values" and v is not None:
                moved[k] = v.to(device, dtype=dtype)
            elif k in ("input_ids","attention_mask","labels") and v is not None:
                moved[k] = v.to(device)
        if batch.get("image_grid_thw") is not None:
            moved["image_grid_thw"] = batch["image_grid_thw"].to(device)
        out = model(**moved)
        n_tokens = (moved["labels"] != -100).sum().item()
        total_loss += out.loss.item() * max(1, n_tokens)
        total_tokens += max(1, n_tokens)
    model.train()
    import math
    return math.exp(total_loss / max(1, total_tokens))

@torch.no_grad()
def eval_gen_accuracy(model, processor, loader, k_samples=50, max_new_tokens=3):
    model.eval()
    correct, seen = 0, 0
    for batch in loader:
        for img, gp, gold in zip(batch["raw_images"], batch["gen_prompts"], batch["answer_text"]):
            if seen >= k_samples: break

            enc = processor(text=[gp], images=[img], return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,                           # <- avoid beam visual expansion issues
                pad_token_id=processor.tokenizer.eos_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                repetition_penalty=1.5,               # discourage "catcatcat..."
                length_penalty=2.0,                   # shorter outputs preferred
                use_cache=True,
            )
            new_tokens = gen_ids[:, enc["input_ids"].shape[1]:]
            text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip().lower()
            pred = text.split()[0] if text else ""
            if gold.lower() in pred:
                correct += 1
            seen += 1
        if seen >= k_samples: break
    model.train()
    return correct / max(1, seen)

print("Running quick baseline eval (pre-prune)...")
ppl_train = eval_loss(model, train_loader)
acc_val0  = eval_gen_accuracy(model, processor, val_loader, k_samples=40)
print(f"Train PPL (pre-prune): {ppl_train:.2f} | Val Gen@1 Acc (pre-prune): {acc_val0:.2%}")


Running quick baseline eval (pre-prune)...


/voc/data/venv-torch2.3/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/voc/data/venv-torch2.3/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/voc/data/venv-torch2.3/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:623: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/voc/data/venv-torch2.3/lib/python3.10/site-packages/transfo

Train PPL (pre-prune): 2244.77 | Val Gen@1 Acc (pre-prune): 0.00%


# 8) Apply Pruning (Pick Mode via Flag)
This cell:
- Applies pruning to the model based on the selected mode (`PRUNE_MODE`):
  - `attn_heads`: Prunes attention heads.
  - `ffn_channels`: Prunes FFN channels.
- Evaluates the model's perplexity after pruning but before fine-tuning.

In [11]:
if PRUNE_MODE == "attn_heads":
    n = 0
    for name, attn in find_attn_modules(model):
        prune_attention_heads_logical_gqa(attn, PRUNE_RATIO)
        n += 1
    print(f"Pruned heads in {n} attention modules (GQA-safe, mask-based).")
elif PRUNE_MODE == "ffn_channels":
    n = 0
    for name, mlp in find_mlp_modules(model):
        prune_ffn_channels_logical_mask(mlp, PRUNE_RATIO, intermediate_size)
        n += 1
    print(f"Pruned channels in {n} MLP modules (mask-based).")
else:
    raise ValueError("PRUNE_MODE must be 'attn_heads' or 'ffn_channels'.")

ppl_after_prune = eval_loss(model, train_loader)
print(f"PPL (post-prune, pre-LoRA): {ppl_after_prune:.2f}")

Pruned channels in 28 MLP modules (mask-based).


PPL (post-prune, pre-LoRA): 677148.78


# 9) LoRA Setup (Attention + MLP Targets)
This cell:
- Identifies target modules for LoRA (Low-Rank Adaptation).
- Configures LoRA parameters such as rank, alpha, and dropout.
- Wraps the model with LoRA layers for fine-tuning.
- Prints the number of trainable parameters after applying LoRA.

In [12]:
def collect_lora_targets(m: nn.Module) -> List[str]:
    names = set()
    for n, mod in m.named_modules():
        if isinstance(mod, nn.Linear):
            for key in ["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj","gate_proj"]:
                if n.endswith(key): names.add(n.split(".")[-1])
    return sorted(list(names)) or ["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj","gate_proj"]

targets = collect_lora_targets(model)
print("LoRA targets:", targets)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=targets
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

LoRA targets: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']


trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290


# 10) Tiny Fine-Tune Loop (Mixed Precision)
This cell:
- Implements a fine-tuning loop with mixed precision (FP16).
- Uses gradient accumulation to simulate a larger batch size.
- Updates the optimizer and learning rate scheduler after each step.
- Prints the loss at each step for monitoring.

In [13]:
# ── Memory & Resource Profiler ────────────────────────────────────────────────
# Run this cell before training to establish a baseline and diagnose crashes.
#
# How to interpret results:
#   GPU free < 1 GB before training  → very high OOM risk during backward/merge
#   loss → NaN with GPU memory stable → numerical instability (NOT OOM)
#   Peak GPU ≈ total GPU at crash     → OOM is the confirmed cause
# ─────────────────────────────────────────────────────────────────────────────
import psutil, gc

def mem_stats(tag=""):
    """Print a labeled snapshot of current GPU + CPU memory usage."""
    if torch.cuda.is_available():
        alloc  = torch.cuda.memory_allocated()  / 1024**3
        reserv = torch.cuda.memory_reserved()   / 1024**3
        total  = torch.cuda.get_device_properties(0).total_memory / 1024**3
        peak   = torch.cuda.max_memory_allocated() / 1024**3
        free   = total - reserv
        print(f"[{tag}] GPU  alloc={alloc:.2f} GB | reserved={reserv:.2f} GB | "
              f"peak={peak:.2f} GB | free={free:.2f} GB / {total:.2f} GB")
    proc      = psutil.Process()
    ram_used  = proc.memory_info().rss / 1024**3
    ram_avail = psutil.virtual_memory().available / 1024**3
    ram_total = psutil.virtual_memory().total / 1024**3
    print(f"[{tag}] CPU  used={ram_used:.2f} GB | avail={ram_avail:.2f} GB / {ram_total:.2f} GB")

# Reset peak counter so training peak reflects only the training phase
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

mem_stats("pre-train baseline")


[pre-train baseline] GPU  alloc=4.16 GB | reserved=4.44 GB | peak=4.16 GB | free=10.12 GB / 14.56 GB


[pre-train baseline] CPU  used=3.50 GB | avail=10.01 GB / 15.33 GB


In [ ]:
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=0.0
)
steps_per_epoch    = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
num_training_steps = EPOCHS * steps_per_epoch
sched = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, num_training_steps)

# GradScaler disabled: PEFT keeps LoRA params in fp16, which GradScaler cannot
# unscale. We rely on gradient clipping below for numerical stability instead.
scaler = torch.cuda.amp.GradScaler(enabled=False)

# ── Key fix: clip gradients to prevent FP16 overflow after aggressive pruning.
# After pruning, PPL can spike to 600K+, producing huge loss gradients.
# Without clipping, the first update corrupts weights → NaN cascades forever.
MAX_GRAD_NORM  = 1.0
MEM_LOG_EVERY  = 25   # log GPU/CPU memory every N optimizer steps

model.train()
global_step = 0
nan_steps   = 0

for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader):
        moved = {}
        for k, v in batch.items():
            if k == "pixel_values":
                moved[k] = v.to(device, dtype=dtype)
            elif k in ("input_ids", "attention_mask", "labels"):
                moved[k] = v.to(device)
        if batch.get("image_grid_thw") is not None:
            moved["image_grid_thw"] = batch["image_grid_thw"].to(device)

        # Pass dtype explicitly so autocast uses bfloat16 (not float16)
        with torch.cuda.amp.autocast(enabled=(dtype != torch.float32), dtype=dtype):
            out  = model(**moved)
            loss = out.loss / GRAD_ACCUM_STEPS

        # ── NaN guard: if loss is already NaN, skip backward to avoid further
        # weight corruption. Root cause is FP16 overflow from the pruned model.
        if not torch.isfinite(loss):
            nan_steps += 1
            if nan_steps <= 3:
                print(f"  step {step+1}: NaN/Inf loss — skipping backward "
                      f"(FP16 overflow after pruning; gradient clipping should prevent this)")
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            # Unscale before clipping (no-op when GradScaler is disabled)
            scaler.unscale_(optimizer)
            # Clip gradients — the primary fix for post-pruning NaN cascades
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=MAX_GRAD_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            sched.step()
            global_step += 1
            print(f"step {global_step:3d} | loss={loss.item() * GRAD_ACCUM_STEPS:.4f}")

            if global_step % MEM_LOG_EVERY == 0:
                mem_stats(f"step-{global_step}")

if nan_steps > 0:
    print(f"\nWarning: {nan_steps}/{len(train_loader)} steps had NaN loss and were skipped.")
    print("  This indicates FP16 overflow after pruning.")
    print("  If still occurring: lower PRUNE_RATIO, use bfloat16, or reduce LR.")

# ── Post-training memory report ──────────────────────────────────────────────
peak_gb  = torch.cuda.max_memory_allocated() / 1024**3
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
mem_stats("post-train")
print(f"\nPeak GPU during training: {peak_gb:.2f} GB / {total_gb:.2f} GB total")
if peak_gb > 0.9 * total_gb:
    print("WARNING: Peak > 90% of GPU — OOM risk is HIGH in the next merge step.")
else:
    print("Peak is within safe range — if a crash occurs next, it is likely OOM at merge.")

torch.cuda.empty_cache()


# 11) Evaluate After LoRA & Save Artifacts
This cell:
- Evaluates the model's perplexity and generation accuracy after fine-tuning.
- Saves the LoRA adapter and the merged model to disk.
- Prints the paths to the saved artifacts.

In [15]:
# ── Free training state before eval/merge to maximise available GPU headroom ─
for _obj in ("optimizer", "sched", "scaler"):
    if _obj in globals():
        del globals()[_obj]
gc.collect()
torch.cuda.empty_cache()
mem_stats("pre-eval (optimizer freed)")

ppl_after_lora = eval_loss(model, train_loader)
acc_val1       = eval_gen_accuracy(model, processor, val_loader, k_samples=40)
print(f"PPL (post-LoRA): {ppl_after_lora:.2f} | Val Gen@1 Acc (post-LoRA): {acc_val1:.2%}")

# Save LoRA adapter (lightweight — always safe regardless of GPU headroom)
model.save_pretrained(os.path.join(OUTPUT_DIR, "lora_adapter"))
processor.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to:", os.path.join(OUTPUT_DIR, "lora_adapter"))

# ── Memory-safe merge ────────────────────────────────────────────────────────
# merge_and_unload() internally calls copy.deepcopy() on the full 4+ GB model,
# which requires an additional ~4 GB of GPU memory and OOMs on this setup.
#
# Instead we use merge_adapter() which fuses LoRA into base weights IN-PLACE
# (no copy), then extract the base model directly. This uses only ~37 MB of
# extra GPU memory (size of the LoRA delta weights), not a full model copy.
gc.collect()
torch.cuda.empty_cache()
mem_stats("pre-merge")

try:
    # Step 1: fuse LoRA deltas into the base weight tensors in-place
    model.merge_adapter()

    # Step 2: pull out the underlying base model — same object, no allocation
    merged = model.base_model.model

    # Step 3: save (serialises GPU tensors to CPU+disk; uses ~4 GB CPU RAM)
    merged.save_pretrained(os.path.join(OUTPUT_DIR, "merged_full"))
    print("Merged checkpoint saved to:", os.path.join(OUTPUT_DIR, "merged_full"))
    mem_stats("post-merge")

except (torch.cuda.OutOfMemoryError, MemoryError, RuntimeError) as e:
    print(f"Merge/save failed ({type(e).__name__}): {e}")
    print("The LoRA adapter above is already saved and sufficient for deployment.")
    print("To deploy: load base model, then apply adapter with PeftModel.from_pretrained().")
    merged = model   # use the PEFT model as-is for the inference demo below


[pre-eval (optimizer freed)] GPU  alloc=4.28 GB | reserved=4.60 GB | peak=4.47 GB | free=9.96 GB / 14.56 GB


[pre-eval (optimizer freed)] CPU  used=3.61 GB | avail=9.90 GB / 15.33 GB


PPL (post-LoRA): nan | Val Gen@1 Acc (post-LoRA): 0.00%


Saved LoRA adapter to: ./demo_pruned_lora_qwen2vl/lora_adapter
[pre-merge] GPU  alloc=4.28 GB | reserved=4.60 GB | peak=4.47 GB | free=9.96 GB / 14.56 GB
[pre-merge] CPU  used=3.74 GB | avail=9.77 GB / 15.33 GB


Merged checkpoint saved to: ./demo_pruned_lora_qwen2vl/merged_full
[post-merge] GPU  alloc=4.28 GB | reserved=4.65 GB | peak=4.47 GB | free=9.91 GB / 14.56 GB
[post-merge] CPU  used=8.64 GB | avail=4.73 GB / 15.33 GB


# 12) Quick Inference Demo (Visual QA)
This cell:
- Demonstrates the model's ability to answer visual questions after fine-tuning.
- Defines a function to generate answers for images and questions.
- Runs the model on a few examples from the validation dataset and prints the predictions.

In [16]:
def qwen_vl_infer(model, processor, pil_image, question: str, max_new_tokens=3):
    pil_image = pil_image.resize((224, 224))
    messages = [{"role":"user","content":[
        {"type":"image","image": pil_image},
        {"type":"text","text": question}
    ]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = processor(text=[prompt], images=[pil_image], return_tensors="pt")
    if "image_grid_thw" in inputs:
        inputs["image_grid_thw"] = inputs["image_grid_thw"]

    for k, v in list(inputs.items()):
        if k == "pixel_values":
            inputs[k] = v.to(device, dtype=dtype)
        else:
            inputs[k] = v.to(device)

    gen_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=processor.tokenizer.eos_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        repetition_penalty=1.5,
        length_penalty=2.0,
        use_cache=True
    )
    new_tokens = gen_ids[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    return text

def demo_val_predictions(model_to_use, k=5):
    # NOTE: test_raw was freed after encoding; use val_items for the demo images
    for i in range(k):
        item = val_items[i]
        ans  = item["answer"]
        pred = qwen_vl_infer(model_to_use, processor, item["image"], "What is in this image?")
        print(f"GT: {ans:<10s} | PRED: {pred}")

print("\n--- Demo predictions (merged model) ---")
demo_val_predictions(merged, k=5)



--- Demo predictions (merged model) ---


/voc/data/venv-torch2.3/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/voc/data/venv-torch2.3/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/voc/data/venv-torch2.3/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:623: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/voc/data/venv-torch2.3/lib/python3.10/site-packages/transfo

GT: cat        | PRED: !!!
GT: ship       | PRED: !!!


GT: ship       | PRED: !!!
GT: airplane   | PRED: !!!


GT: frog       | PRED: !!!


In [17]:
# Empty cell for additional code or notes.

In [18]:
# Empty cell for additional code or notes.